<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase33_COLAB_DOWNLOAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 33
## Person-disjoint, label-masked synthetic challenge of imaging evidence preservation
**Google Drive → Google Drive** · synthetic only · no Athena file required

### Entry gate
Phase 32 reported 12/12 oracle-anchored **debug** cases on a three-person synthetic fixture (23 source observations); it did not provide an independent, blinded evaluation, a faithful Arm A, or official OMOP/MI-CDM cohort equivalence.

### New experiment
1. Read and SHA-256 verify the **matching** Phase 32 JSON, expanded FHIR bundle, case CSV, event CSV and source-proxy cohort CSV from the same Drive folder.
2. **Freeze a detector module before generating any Phase 33 cases**, save its checksum, then create a new, person-disjoint, fully synthetic FHIR-style fixture with 19 new persons, 38 studies and 148 source observations. Four persons are deliberately source-proxy negative controls.
3. Generate 19 deterministic test scenarios and keep their injection labels out of the detector call. Compare a deliberately *value-only ablation* against a **source-manifest imaging-contract prototype**. Neither is the completed standards-supported Arm A or integrated Arm D.
4. Reveal ground-truth labels after prediction; report event-level TP/FP/FN, false alerts, case-level results, source-proxy membership disagreement and negative controls. These tests are not externally blinded: the same notebook constructs the corruptions. They are not clinical sensitivity/specificity estimates.

**Blocked gates stay blocked:** original generic Arm C is nonportable; Athena snapshot and 19 reviewed bindings are absent; 36 original vocabulary gaps remain. No real ADNI rows, pyOMOP ETL, official MI-CDM load, MCI attestation or four-arm comparison.

**Publication caution:** New synthetic person IDs and label masking improve separation from Phase 32, but the researcher controls case generation and detector definitions. An independently authored held-out set is still required for strong performance claims.

In [1]:
# Colab mount; an isolated path override is provided only for notebook regression tests.
import os
DRIVE_ROOT_OVERRIDE = os.environ.get('ICHI2027_DRIVE_ROOT', '').strip()
if not DRIVE_ROOT_OVERRIDE:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import json, hashlib, copy, csv, uuid, random, importlib.util
from collections import defaultdict, Counter
from datetime import datetime, timezone, timedelta
import pandas as pd

DRIVE_ROOT = Path(DRIVE_ROOT_OVERRIDE) if DRIVE_ROOT_OVERRIDE else Path('/content/drive/MyDrive')
PREFERRED_PHASE32_DIR = ''  # Optional: '/content/drive/MyDrive/your-folder'. Leave blank to discover.

NAMES = {
  'bundle':'ICHI2027_Phase32_EXPANDED_SYNTHETIC_FHIR_BUNDLE.json',
  'cases':'ICHI2027_Phase32_CASE_RESULTS.csv',
  'events':'ICHI2027_Phase32_EVENT_RESULTS.csv',
  'cohort':'ICHI2027_Phase32_SOURCE_PROXY_COHORT.csv'
}
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def safe_files(folder, pattern):
    try: it=folder.rglob(pattern)
    except OSError: return []
    ans=[]
    try:
        for p in it:
            try:
                if p.is_file(): ans.append((p.stat().st_mtime,p))
            except (OSError,FileNotFoundError): pass  # Google Drive can expose stale entries
    except OSError:pass
    return [p for _,p in sorted(ans,reverse=True)]
def sibling_match(folder, basename, expected):
    stem=Path(basename).stem
    suffix=Path(basename).suffix
    candidates=safe_files(folder,stem+'*'+suffix)
    for p in candidates:
        if p.name==basename or p.name.startswith(stem+'(') or p.name.startswith(stem+'_'):
            try:
                if sha(p)==expected:return p
            except (OSError,FileNotFoundError):pass
    return None

assert DRIVE_ROOT.is_dir(), f'Drive root unavailable: {DRIVE_ROOT}'
search_root=Path(PREFERRED_PHASE32_DIR) if PREFERRED_PHASE32_DIR else DRIVE_ROOT
candidates=safe_files(search_root,'ICHI2027_Phase32_SHAREABLE*.json')
selected=None; failures=[]
for candidate in candidates:
    try:
        payload=json.loads(candidate.read_text(encoding='utf-8'))
        if payload.get('phase')!='32_source_manifest_controlled_synthetic_evidence_loss_challenges':continue
        expected=dict(payload['public_output_sha256'])
        expected['bundle']=payload['synthetic_fixture']['expanded_bundle_sha256']
        found={key:sibling_match(candidate.parent,name,expected[key]) for key,name in NAMES.items()}
        if any(x is None for x in found.values()):
            failures.append(str(candidate)+': matching bundle/CSV hash not found in SAME directory')
            continue
        selected=(candidate,payload,found);break
    except (ValueError,KeyError,FileNotFoundError,OSError) as exc:
        failures.append(str(candidate)+': '+repr(exc))
if selected is None:
    raise FileNotFoundError('No complete hash-matching Phase 32 input set in Drive. '+
       'Check that SHAREABLE, expanded bundle, CASE_RESULTS, EVENT_RESULTS and SOURCE_PROXY_COHORT '+
       'are in the same Drive folder. Previous candidates: '+repr(failures[:5]))
P32_JSON,p32,P32_FILES=selected
WORKDIR=P32_JSON.parent
assert p32['experiments']['cases_passing_declared_expectations']==12
assert p32['experiments']['independent_heldout_validation'] is False
assert p32['athena_gate']['original_vocab_gaps_remaining_blocked']==36
assert p32['gates']['synthetic_faithful_armA_mapping_completed'] is False
assert p32['gates']['official_target_cohort_equivalence_tested'] is False
with open(P32_FILES['bundle'],encoding='utf-8') as f:old_bundle=json.load(f)
assert old_bundle.get('resourceType')=='Bundle'
assert sum(e['resource'].get('resourceType')=='Observation' for e in old_bundle['entry'])==23
assert len(pd.read_csv(P32_FILES['cases']))==12
assert len(pd.read_csv(P32_FILES['cohort']))==36
print('Verified same-folder Phase 32 inputs:')
for key,p in {'shareable':P32_JSON,**P32_FILES}.items():print(' ',key,p.name)
print('Phase 33 output folder:',WORKDIR)

Verified same-folder Phase 32 inputs:
  shareable ICHI2027_Phase32_SHAREABLE.json
  bundle ICHI2027_Phase32_EXPANDED_SYNTHETIC_FHIR_BUNDLE.json
  cases ICHI2027_Phase32_CASE_RESULTS.csv
  events ICHI2027_Phase32_EVENT_RESULTS.csv
  cohort ICHI2027_Phase32_SOURCE_PROXY_COHORT.csv
Phase 33 output folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


## 1. Freeze detector **before** new case generation
The detector below accepts only the clean **source evidence manifest**, candidate transformed-shadow observations and study ownership. Injected fault labels are never passed to it. Comparing transformation output to an authorized source manifest is intrinsic to a fidelity audit; it is not the same as supplying engineered-fault labels.

The module's SHA-256 is checked again before export. Once generated, do not edit the detector and still call the run a frozen evaluation.

In [3]:
DETECTOR_SOURCE = r'''
from collections import Counter
ALLOWED_UNITS={'mm3':1.0,'cm3':1000.0}

def detect(source_rows, target_rows, study_owner):
    # Never receives any fault labels or an expected-case identifier.
    by_source={r['observation_id']:r for r in source_rows}
    by_target={}
    for r in target_rows: by_target.setdefault(r['observation_id'],[]).append(r)
    alerts=set()
    def emit(oid,code): alerts.add((oid,code))
    for oid,src in by_source.items():
        found=by_target.get(oid,[])
        if not found:
            emit(oid,'SOURCE_OBSERVATION_MISSING');continue
        if len(found)>1:emit(oid,'DUPLICATE_OBSERVATION_ID')
        # Auditing first copy does not excuse duplicate: its own alert remains.
        row=found[0]
        if row.get('patient_ref')!=src['patient_ref']:emit(oid,'PATIENT_MISMATCH')
        target_study=row.get('study_ref')
        if target_study!=src['study_ref']:emit(oid,'STUDY_LINK_MISMATCH')
        if target_study not in study_owner:emit(oid,'SOURCE_LINK_UNRESOLVED')
        elif study_owner[target_study]!=src['patient_ref']:emit(oid,'CROSS_PERSON_STUDY_LINK')
        if (row.get('anatomy_code')!=src['anatomy_code']):emit(oid,'ANATOMY_LATERALITY_MISMATCH')
        unit=row.get('unit')
        if unit in (None,''):emit(oid,'UNIT_MISSING')
        elif unit not in ALLOWED_UNITS:emit(oid,'UNIT_INCOMPATIBLE')
        else:
            try:
                v=float(row['value'])*ALLOWED_UNITS[unit]
                original=float(src['value'])*ALLOWED_UNITS[src['unit']]
                if abs(v-original)>1e-6:emit(oid,'VALUE_MISMATCH')
            except (TypeError,ValueError,KeyError):emit(oid,'VALUE_MISMATCH')
        if not row.get('algorithm_version'):emit(oid,'MODEL_VERSION_LOSS')
        elif row['algorithm_version']!=src['algorithm_version']:emit(oid,'MODEL_VERSION_MISMATCH')
        if not row.get('trace_state'):emit(oid,'TRACE_STATE_LOSS')
        elif row['trace_state']!=src['trace_state']:emit(oid,'TRACE_STATE_MISMATCH')
        if not row.get('provenance_ok'):emit(oid,'PROVENANCE_CHAIN_LOSS')
    for oid in by_target:
        if oid not in by_source:emit(oid,'UNEXPECTED_TARGET_FEATURE')
    return [{'observation_id':oid,'warning_code':code} for oid,code in sorted(alerts)]

def value_only_member(rows):
    # Deliberate B-style *ablation* proxy; not a faithful standards-based comparator.
    slots=set()
    for r in rows:
        if (r.get('region')!='hippocampus' or r.get('laterality') not in ('left','right')
                or r.get('timepoint') not in ('baseline','12-month')):continue
        try:positive=float(r.get('value'))>0
        except (TypeError,ValueError):positive=False
        if positive:slots.add((r['region'],r['laterality'],r['timepoint']))
    return len(slots)==4

def qualified_member(rows,alerts):
    flagged={x['observation_id'] for x in alerts}
    return value_only_member([r for r in rows if r['observation_id'] not in flagged])
'''
from pathlib import Path
import hashlib, importlib.util
MODULE_PATH=Path('/content/phase33_detector_frozen.py') if not DRIVE_ROOT_OVERRIDE else Path('/tmp/phase33_detector_frozen.py')
MODULE_PATH.write_text(DETECTOR_SOURCE,encoding='utf-8')
DETECTOR_SHA=hashlib.sha256(MODULE_PATH.read_bytes()).hexdigest()
spec=importlib.util.spec_from_file_location('phase33_detector_frozen',str(MODULE_PATH))
frozen=importlib.util.module_from_spec(spec);spec.loader.exec_module(frozen)
assert DETECTOR_SHA==hashlib.sha256(MODULE_PATH.read_bytes()).hexdigest()
print('Detector frozen SHA-256:',DETECTOR_SHA)

Detector frozen SHA-256: 91207ce50b97f0e195aabfe823b4be25886a21bf7300b344fab03c75bced395d


## 2. Generate fresh, person-disjoint synthetic FHIR evidence
All persons, study/series identifiers, observation IDs and dates below are newly generated synthetic examples, not ADNI records. Person separation is from Phase 32's three identities; the same investigator still designs the challenge families. The four deliberate source-proxy-negative controls lack a source right-hippocampus 12-month observation.

The FHIR-shaped bundle is for reproducible fixture development. It has **not** passed an external HL7 profile validator and it has **no MCI labels**.

In [4]:
SEED=202733
rng=random.Random(SEED)
SYSTEM='https://example.org/ichi2027/phase33/synthetic'
UCUM='http://unitsofmeasure.org'
REGION_CODE={'hippocampus':'SYN-HIP','entorhinal':'SYN-ENT'}
PERSON_IDS=[f'held33-p{i:03d}' for i in range(1,20)]
NEGATIVE=set(PERSON_IDS[-4:])

# Deterministic, syntactically plausible test-only 2.25 OIDs; not real DICOM UIDs.
def synthetic_oid(label):return '2.25.'+str(uuid.uuid5(uuid.NAMESPACE_URL,SYSTEM+'/'+label).int)
def reference(rt,rid):return {'reference':f'{rt}/{rid}'}
def extension(url,key,value):return {'url':SYSTEM+'/StructureDefinition/'+url,key:value}

entries=[]
for i,pid in enumerate(PERSON_IDS):
    entries.append({'resource':{'resourceType':'Patient','id':pid,
        'identifier':[{'system':SYSTEM+'/patient','value':pid}]}})
    dev=f'dev-{pid}'
    entries.append({'resource':{'resourceType':'Device','id':dev,
        'deviceName':[{'name':'SynthSegLike','type':'user-friendly-name'}],
        'version':[{'value':'2.1'}]}})
    for timepoint in ('baseline','12-month'):
        v='bl' if timepoint=='baseline' else 'm12'
        dt=(datetime(2024 if v=='bl' else 2025,3,14,tzinfo=timezone.utc)+timedelta(days=i)).isoformat().replace('+00:00','Z')
        sid=f'study-{pid}-{v}'
        entries.append({'resource':{'resourceType':'ImagingStudy','id':sid,'status':'available',
          'subject':reference('Patient',pid),'started':dt,
          'identifier':[{'system':'urn:dicom:uid','value':'urn:oid:'+synthetic_oid(sid)}],
          'series':[{'uid':synthetic_oid(sid+'/series'),
              'modality':{'system':'http://dicom.nema.org/resources/ontology/DCM','code':'MR'}}]}})
        for region in ('hippocampus','entorhinal'):
            for side in ('left','right'):
                if pid in NEGATIVE and region=='hippocampus' and side=='right' and v=='m12':continue
                oid=f'obs-{pid}-{region}-{side}-{v}'
                base=(3550 if region=='hippocampus' else 1750)+(105 if side=='right' else 0)+i*13
                amount=base-(83 if region=='hippocampus' else 32 if v=='m12' else 0) if v=='m12' else base
                amount=amount+rng.randint(-25,25)
                obs={'resourceType':'Observation','id':oid,'status':'final',
                  'subject':reference('Patient',pid),'focus':[reference('ImagingStudy',sid)],
                  'device':reference('Device',dev),
                  'code':{'coding':[{'system':SYSTEM+'/biomarker','code':region+'-volume'}]},
                  'bodySite':{'coding':[{'system':SYSTEM+'/anatomy','code':REGION_CODE[region]+'-'+side.upper()}]},
                  'effectiveDateTime':dt,
                  'valueQuantity':{'value':amount,'unit':'mm3','system':UCUM,'code':'mm3'},
                  'extension':[extension('timepoint-label','valueString',timepoint),
                               extension('trace-state','valueCode','PASS'),
                               extension('trace-rule-version','valueString','TRACE-synthetic-v2')]}
                entries.append({'resource':obs})
                entries.append({'resource':{'resourceType':'Provenance','id':'prov-'+oid,
                   'recorded':'2026-09-28T00:00:00Z',
                   'target':[reference('Observation',oid)],
                   'entity':[{'role':'source','what':reference('ImagingStudy',sid)},
                             {'role':'source','what':reference('Device',dev)}]}})
heldout_bundle={'resourceType':'Bundle','type':'collection','identifier':{
   'system':SYSTEM+'/bundle','value':'phase33-independent-synthetic-persons'},'entry':entries}
counts=Counter(e['resource']['resourceType'] for e in entries)
assert counts['Patient']==19 and counts['ImagingStudy']==38
assert counts['Observation']==148 and counts['Provenance']==148
assert not set(PERSON_IDS)&{'syn-p01','syn-p02','aug-syn-p03'}
print('Fresh synthetic FHIR resource counts:',dict(counts))

Fresh synthetic FHIR resource counts: {'Patient': 19, 'Device': 19, 'ImagingStudy': 38, 'Observation': 148, 'Provenance': 148}


In [5]:
# Extract source evidence directly from newly generated FHIR resources.
residx={f"{e['resource']['resourceType']}/{e['resource']['id']}":e['resource']
        for e in heldout_bundle['entry']}
study_owner={f'ImagingStudy/{x["id"]}':x['subject']['reference']
             for x in residx.values() if x['resourceType']=='ImagingStudy'}
provenance_target=set()
for resource in residx.values():
    if resource['resourceType']=='Provenance':
        for target in resource.get('target',[]):provenance_target.add(target['reference'])
def ext(r,suffix,key):
    return next((e.get(key) for e in r.get('extension',[])
                if e.get('url','').endswith('/'+suffix)),None)
def normalize(obs):
    oid=obs['id']
    focus=obs.get('focus',[{}])[0].get('reference')
    dref=obs.get('device',{}).get('reference')
    dev=residx.get(dref,{})
    coded=obs.get('bodySite',{}).get('coding',[{}])[0].get('code','')
    region='hippocampus' if 'HIP-' in coded else 'entorhinal' if 'ENT-' in coded else None
    side='left' if coded.endswith('-LEFT') else 'right' if coded.endswith('-RIGHT') else None
    return {'observation_id':oid,'patient_ref':obs['subject']['reference'],
        'study_ref':focus,'device_ref':dref,'region':region,'laterality':side,
        'anatomy_code':coded,'timepoint':ext(obs,'timepoint-label','valueString'),
        'value':obs.get('valueQuantity',{}).get('value'),
        'unit':obs.get('valueQuantity',{}).get('unit'),
        'algorithm_version':dev.get('version',[{}])[0].get('value') if dev.get('version') else None,
        'trace_state':ext(obs,'trace-state','valueCode'),
        'provenance_ok':f'Observation/{oid}' in provenance_target}
SOURCE_ROWS=[normalize(r) for r in residx.values() if r['resourceType']=='Observation']
assert len(SOURCE_ROWS)==148
source_by_patient=defaultdict(list)
for r in SOURCE_ROWS:source_by_patient[r['patient_ref']].append(r)
for pid in PERSON_IDS:
    rows=source_by_patient['Patient/'+pid]
    assert frozen.value_only_member(rows)==(pid not in NEGATIVE)
print('Source-proxy cohort:',sum(frozen.value_only_member(rows)
      for rows in source_by_patient.values()),'eligible / 19 synthetic people')

Source-proxy cohort: 15 eligible / 19 synthetic people


## 3. Create label-masked challenge cases
Case IDs and corruption types below are used **only for generation and post-detection adjudication**. The frozen detector receives no case IDs, corruption instructions or sealed expected warnings. A successful cell here is still *developer-controlled synthetic test performance*, not independent external validation.

In [6]:
# 19 unique case subjects, each disjoint from all Phase 32 persons.
scenario_specs=[
  ('H00_clean_positive',PERSON_IDS[0],'clean'),
  ('H01_clean_negative',PERSON_IDS[15],'clean'),
  ('H02_valid_cm3_conversion',PERSON_IDS[1],'valid_cm3'),
  ('H03_bilateral_label_swap',PERSON_IDS[2],'swap'),
  ('H04_unit_missing',PERSON_IDS[3],'missing_unit'),
  ('H05_unit_incompatible',PERSON_IDS[4],'bad_unit'),
  ('H06_wrong_time_same_person',PERSON_IDS[5],'wrong_time'),
  ('H07_cross_person_study',PERSON_IDS[6],'cross_person'),
  ('H08_algorithm_version_missing',PERSON_IDS[7],'model_version'),
  ('H09_trace_state_missing',PERSON_IDS[8],'trace'),
  ('H10_duplicate_observation',PERSON_IDS[9],'duplicate'),
  ('H11_provenance_missing',PERSON_IDS[10],'provenance'),
  ('H12_fabricated_false_inclusion',PERSON_IDS[16],'fabricated'),
  ('H13_missing_source_observation',PERSON_IDS[11],'missing_obs'),
  ('H14_combined_heldout_fault',PERSON_IDS[12],'combined'),
  ('H15_second_clean_positive',PERSON_IDS[13],'clean'),
  ('H16_second_clean_negative',PERSON_IDS[17],'clean'),
  ('H17_entorhinal_noncohort_fault',PERSON_IDS[14],'noncohort'),
  ('H18_third_clean_negative',PERSON_IDS[18],'clean')]

def matched(rows,region='hippocampus',side='left',visit='baseline'):
    matches=[r for r in rows if (r['region'],r['laterality'],r['timepoint'])==(region,side,visit)]
    assert len(matches)==1,(region,side,visit)
    return matches[0]
def make_case(pid,kind):
    original=copy.deepcopy(source_by_patient['Patient/'+pid])
    target=copy.deepcopy(original)
    labels=set()
    def truth(row,code):labels.add((row['observation_id'],code))
    if kind=='valid_cm3':
        r=matched(target)
        r['value']=r['value']/1000.0;r['unit']='cm3'
    elif kind=='swap':
        l=matched(target,side='left');r=matched(target,side='right')
        l['anatomy_code'],r['anatomy_code']=r['anatomy_code'],l['anatomy_code']
        truth(l,'ANATOMY_LATERALITY_MISMATCH');truth(r,'ANATOMY_LATERALITY_MISMATCH')
    elif kind=='missing_unit':
        r=matched(target,side='right',visit='12-month');r['unit']=None;truth(r,'UNIT_MISSING')
    elif kind=='bad_unit':
        r=matched(target);r['unit']='kg';truth(r,'UNIT_INCOMPATIBLE')
    elif kind=='wrong_time':
        r=matched(target)
        r['study_ref']=r['study_ref'].replace('-bl','-m12');truth(r,'STUDY_LINK_MISMATCH')
    elif kind=='cross_person':
        r=matched(target,side='right',visit='12-month')
        r['study_ref']='ImagingStudy/study-'+PERSON_IDS[7]+'-m12'
        truth(r,'STUDY_LINK_MISMATCH');truth(r,'CROSS_PERSON_STUDY_LINK')
    elif kind=='model_version':
        for r in target:r['algorithm_version']=None;truth(r,'MODEL_VERSION_LOSS')
    elif kind=='trace':
        r=matched(target);r['trace_state']=None;truth(r,'TRACE_STATE_LOSS')
    elif kind=='duplicate':
        r=matched(target);target.append(copy.deepcopy(r));truth(r,'DUPLICATE_OBSERVATION_ID')
    elif kind=='provenance':
        r=matched(target,side='right');r['provenance_ok']=False;truth(r,'PROVENANCE_CHAIN_LOSS')
    elif kind=='fabricated':
        # The source-negative patient lacks right hippocampus at follow-up.
        r=copy.deepcopy(matched(target,side='right',visit='baseline'))
        r['observation_id']='fabricated-'+pid+'-hip-right-m12'
        r['timepoint']='12-month';r['study_ref']=r['study_ref'].replace('-bl','-m12')
        target.append(r);truth(r,'UNEXPECTED_TARGET_FEATURE')
    elif kind=='missing_obs':
        r=matched(target,side='right',visit='12-month')
        target.remove(r);truth(r,'SOURCE_OBSERVATION_MISSING')
    elif kind=='combined':
        r=matched(target)
        r['anatomy_code']='SYN-HIP-RIGHT';r['unit']=None;r['trace_state']=None
        for c in ['ANATOMY_LATERALITY_MISMATCH','UNIT_MISSING','TRACE_STATE_LOSS']:truth(r,c)
    elif kind=='noncohort':
        r=matched(target,region='entorhinal',side='left')
        r['anatomy_code']='SYN-ENT-RIGHT';truth(r,'ANATOMY_LATERALITY_MISMATCH')
    elif kind!='clean':raise ValueError(kind)
    return {'person_ref':'Patient/'+pid,'source_rows':original,'shadow_rows':target},labels

sealed_truth={}
blinded_cases={}
for case,pid,kind in scenario_specs:
    blinded_cases[case],sealed_truth[case]=make_case(pid,kind)
assert len(blinded_cases)==19
assert len({d['person_ref'] for d in blinded_cases.values()})==19
assert DETECTOR_SHA==hashlib.sha256(MODULE_PATH.read_bytes()).hexdigest()
print('19 new synthetic challenge cases prepared; labels sealed outside the detector arguments.')

19 new synthetic challenge cases prepared; labels sealed outside the detector arguments.


## 4. Run frozen audit with labels concealed, then unseal once
The source proxy asks only whether each person has baseline and 12-month **bilateral hippocampus volume evidence**. It does not attest MCI, cognition or clinical eligibility. A qualified exclusion resulting from a flagged corruption is **a review/withholding decision**, not proof of an unintended cohort error.

In [7]:
# No case labels or expectations are passed to detect().
blind_predictions={}
for case,data in blinded_cases.items():
    alerts=frozen.detect(data['source_rows'],data['shadow_rows'],study_owner)
    source_member=frozen.value_only_member(data['source_rows'])
    b_member=frozen.value_only_member(data['shadow_rows'])
    d_member=frozen.qualified_member(data['shadow_rows'],alerts)
    blind_predictions[case]={'alerts':alerts,'source_member':source_member,
        'value_only_member':b_member,'qualified_member':d_member}

# Once all predictions exist, reveal injected synthetic labels for scoring.
assert set(blind_predictions)==set(sealed_truth)
case_rows=[];event_rows=[];cohort_rows=[];label_rows=[]
for case,pid,kind in scenario_specs:
    pred=blind_predictions[case]
    observed={(x['observation_id'],x['warning_code']) for x in pred['alerts']}
    truth=sealed_truth[case]
    tp=len(observed&truth);fp=len(observed-truth);fn=len(truth-observed)
    source=pred['source_member'];b=pred['value_only_member'];d=pred['qualified_member']
    case_rows.append({'case':case,'scenario_kind':kind,'synthetic_person_ref':'Patient/'+pid,
      'expected_event_count':len(truth),'observed_event_count':len(observed),
      'tp':tp,'fp':fp,'fn':fn,'exact_event_match':observed==truth,
      'source_proxy_member':source,'value_only_proxy_member':b,'qualified_contract_proxy_member':d,
      'value_only_false_inclusion':int(b and not source),'value_only_mismatch':int(b!=source),
      'qualified_policy_withholding':int(source and not d),'qualified_false_inclusion':int(d and not source),
      'qualified_mismatch_vs_clean_source':int(d!=source),
      'independent_external_blinding':False,'official_omop_cohort_equivalence':False})
    for oid,warn in sorted(observed|truth):
        event_rows.append({'case':case,'observation_id':oid,'warning_code':warn,
           'ground_truth_injected':(oid,warn) in truth,'detector_flagged':(oid,warn) in observed,
           'classification':'TP' if (oid,warn) in observed&truth else 'FP' if (oid,warn) in observed else 'FN'})
    for oid,warn in sorted(truth):label_rows.append({'case':case,'observation_id':oid,'injected_warning_code':warn})
    cohort_rows.append({'case':case,'synthetic_person_ref':'Patient/'+pid,
       'source_proxy_member':source,'value_only_shadow_member':b,
       'contract_qualified_shadow_member':d,'MCI_status':'UNKNOWN_NOT_ATTESTED'})
cases_df=pd.DataFrame(case_rows);events_df=pd.DataFrame(event_rows)
labels_df=pd.DataFrame(label_rows);cohort_df=pd.DataFrame(cohort_rows)
assert len(cases_df)==19 and cases_df['exact_event_match'].all()
assert int(cases_df['fn'].sum())==0 and int(cases_df['fp'].sum())==0
assert int(cases_df['value_only_false_inclusion'].sum())==1
assert int(cases_df['qualified_false_inclusion'].sum())==0
assert cases_df.loc[cases_df.case=='H02_valid_cm3_conversion','observed_event_count'].iloc[0]==0
assert cases_df.loc[cases_df.case=='H17_entorhinal_noncohort_fault','qualified_contract_proxy_member'].iloc[0]
print('Event TP/FP/FN (engineered synthetic labels):',
      int(cases_df.tp.sum()),int(cases_df.fp.sum()),int(cases_df.fn.sum()))
print('Exact case matches:',int(cases_df.exact_event_match.sum()),'/',len(cases_df))
print('Value-only false inclusions:',int(cases_df.value_only_false_inclusion.sum()))
print('Qualified exclusions of originally eligible synthetic people:',int(cases_df.qualified_policy_withholding.sum()))
display(cases_df[['case','expected_event_count','observed_event_count','tp','fp','fn',
   'source_proxy_member','value_only_proxy_member','qualified_contract_proxy_member']])

Event TP/FP/FN (engineered synthetic labels): 24 0 0
Exact case matches: 19 / 19
Value-only false inclusions: 1
Qualified exclusions of originally eligible synthetic people: 11


,case,expected_event_count,observed_event_count,tp,fp,fn,source_proxy_member,value_only_proxy_member,qualified_contract_proxy_member
0,H00_clean_positive,0,0,0,0,0,True,True,True
1,H01_clean_negative,0,0,0,0,0,False,False,False
2,H02_valid_cm3_conversion,0,0,0,0,0,True,True,True
3,H03_bilateral_label_swap,2,2,2,0,0,True,True,False
4,H04_unit_missing,1,1,1,0,0,True,True,False
5,H05_unit_incompatible,1,1,1,0,0,True,True,False
6,H06_wrong_time_same_person,1,1,1,0,0,True,True,False
7,H07_cross_person_study,2,2,2,0,0,True,True,False
8,H08_algorithm_version_missing,8,8,8,0,0,True,True,False
9,H09_trace_state_missing,1,1,1,0,0,True,True,False


## 5. Consistency controls and export
Checks compare the run with the frozen detector hash and the independently generated source-proxy labels. We do **not** estimate clinical specificity, patient-level confidence intervals or four-arm performance using these engineered records.

Public-safe files contain synthetic example identifiers only. The official Arm A/MI-CDM, Athena binding and real ADNI gates remain closed regardless of synthetic test success.

In [8]:
assert DETECTOR_SHA==hashlib.sha256(MODULE_PATH.read_bytes()).hexdigest(), 'Frozen detector changed during test.'
assert len([p for p in PERSON_IDS if p not in NEGATIVE])==15
assert len([p for p in PERSON_IDS if p in NEGATIVE])==4
assert not any(pid in ('syn-p01','syn-p02','aug-syn-p03') for pid in PERSON_IDS)
assert not p32['gates']['official_target_cohort_equivalence_tested']
assert not p32['gates']['synthetic_faithful_armA_mapping_completed']
# Every synthetic case uses a different person, but all families are authored by the same developer.
assert len(set(cases_df.synthetic_person_ref))==len(cases_df)
assert cases_df.loc[cases_df.scenario_kind=='clean','observed_event_count'].eq(0).all()
assert cases_df.loc[cases_df.case=='H12_fabricated_false_inclusion','value_only_false_inclusion'].iloc[0]==1
print('Regression controls all passed; the detector remained unchanged through case evaluation.')

Regression controls all passed; the detector remained unchanged through case evaluation.


In [9]:
# Write immutable-named Drive artifacts together with the full exact-input hash chain.
OUTPUTS={
 'shareable':WORKDIR/'ICHI2027_Phase33_SHAREABLE.json',
 'cases':WORKDIR/'ICHI2027_Phase33_HELDOUT_CASE_RESULTS.csv',
 'events':WORKDIR/'ICHI2027_Phase33_HELDOUT_EVENT_RESULTS.csv',
 'cohort':WORKDIR/'ICHI2027_Phase33_SOURCE_PROXY_COHORT.csv',
 'labels':WORKDIR/'ICHI2027_Phase33_INJECTED_SYNTHETIC_LABELS.csv',
 'bundle':WORKDIR/'ICHI2027_Phase33_NEW_SYNTHETIC_FHIR_BUNDLE.json',
 'detector':WORKDIR/'ICHI2027_Phase33_FROZEN_DETECTOR.py',
}
for name,df in [('cases',cases_df),('events',events_df),('cohort',cohort_df),('labels',labels_df)]:
    df.to_csv(OUTPUTS[name],index=False)
OUTPUTS['bundle'].write_text(json.dumps(heldout_bundle,indent=2),encoding='utf-8')
OUTPUTS['detector'].write_bytes(MODULE_PATH.read_bytes())
assert sha(OUTPUTS['detector'])==DETECTOR_SHA
summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'33_person_disjoint_label_masked_synthetic_evidence_audit',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Fully synthetic/public-safe; no ADNI, original patient IDs, or private Athena files.',
 'upstream':{'phase32_shareable_sha256':sha(P32_JSON),
             'phase32_expanded_bundle_sha256':sha(P32_FILES['bundle']),
             'phase32_case_results_sha256':sha(P32_FILES['cases']),
             'phase32_event_results_sha256':sha(P32_FILES['events']),
             'phase32_cohort_sha256':sha(P32_FILES['cohort']),
             'original_athena_binding_keys':19,'unreviewed_athena_gaps':36,
             'imagewg_commit':p32['upstream']['imagewg_pinned_commit'],
             'omop55_commit':p32['upstream']['omop55_pinned_commit']},
 'method':{'detector_frozen_before_new_cases':True,'detector_sha256':DETECTOR_SHA,
           'phase32_person_ids_reused':False,'case_labels_passed_to_detector':False,
           'developer_controls_both_detector_and_case_generation':True,
           'independent_external_blinding':False,
           'comparison_scope':'source-manifest audit prototype vs deliberate value-only shadow ablation',
           'official_omop_or_micdm_comparison':False},
 'synthetic_fixture':{'person_count':19,'source_proxy_positive':15,'source_proxy_negative':4,
                      'imaging_study_count':38,'source_observation_count':148,'resource_counts':dict(counts),
                      'MCI_status_attested':False},
 'evaluation':{'case_count':int(len(cases_df)),'cases_exact_warning_match':int(cases_df.exact_event_match.sum()),
               'injected_events_tp':int(cases_df.tp.sum()),'nonmatching_alerts_fp':int(cases_df.fp.sum()),
               'injected_events_fn':int(cases_df.fn.sum()),
               'value_only_false_inclusion_synthetic_cases':int(cases_df.value_only_false_inclusion.sum()),
               'qualified_false_inclusion_synthetic_cases':int(cases_df.qualified_false_inclusion.sum()),
               'qualified_policy_withholding_synthetic_cases':int(cases_df.qualified_policy_withholding.sum()),
               'clinical_performance_estimated':False,'independently_authored_heldout_validation':False},
 'gates':{'phase32_hash_chain_verified':True,'new_person_disjoint_synthetic_fixture_generated':True,
          'detector_freeze_hash_verified':True,'label_masked_synthetic_challenges_passed':bool(cases_df.exact_event_match.all()),
          'frozen_private_athena_snapshot_provided':False,'all_19_bindings_independently_reviewed':False,
          'original_36_vocabulary_gaps_resolved':False,'synthetic_faithful_armA_mapping_completed':False,
          'original_generic_armC_executed':False,'pyOMOP_ETL_executed':False,
          'official_MICDM_load_completed':False,'real_MRI_rows_processed':0,
          'true_four_arm_comparison_completed':False,'official_target_cohort_equivalence_tested':False},
 'artifact_sha256':{k:sha(p) for k,p in OUTPUTS.items() if k!='shareable'},
 'next_stage_gate':'Secure frozen private Athena CONCEPT.csv, manifest and 19 reviewed source-supported binding decisions to build faithful Arm A; commission truly independent blinded corruption cases to corroborate synthetic detection beyond developer-controlled tests. Then test cardinality and OMOP/MI-CDM cohort equivalence; never call a source-proxy test clinical validation.'
}
OUTPUTS['shareable'].write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Saved Drive→Drive outputs:')
for k,p in OUTPUTS.items():print(' ',k,p)

Saved Drive→Drive outputs:
  shareable /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase33_SHAREABLE.json
  cases /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase33_HELDOUT_CASE_RESULTS.csv
  events /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase33_HELDOUT_EVENT_RESULTS.csv
  cohort /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase33_SOURCE_PROXY_COHORT.csv
  labels /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase33_INJECTED_SYNTHETIC_LABELS.csv
  bundle /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase33_NEW_SYNTHETIC_FHIR_BUNDLE.json
  detector /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase33_FROZEN_DETECTOR.py


In [10]:
# Final Drive read-back verification. Exact hashes of every output are retained in the JSON.
with open(OUTPUTS['shareable'],encoding='utf-8') as f:check=json.load(f)
for key,expected in check['artifact_sha256'].items():
    assert sha(OUTPUTS[key])==expected,(key,'artifact hash mismatch')
assert int(pd.read_csv(OUTPUTS['cases']).exact_event_match.sum())==19
assert len(pd.read_csv(OUTPUTS['labels']))==check['evaluation']['injected_events_tp']
assert check['gates']['official_MICDM_load_completed'] is False
assert check['gates']['official_target_cohort_equivalence_tested'] is False
print('PHASE 33 COMPLETE — all new engineered synthetic checks passed.')
print('Athena-dependent Arm A and official target cohort testing remain blocked.')
print('Share these three PUBLIC-SAFE files for Phase 34:')
for k in ['shareable','cases','events']:print(' -',OUTPUTS[k].name)

PHASE 33 COMPLETE — all new engineered synthetic checks passed.
Athena-dependent Arm A and official target cohort testing remain blocked.
Share these three PUBLIC-SAFE files for Phase 34:
 - ICHI2027_Phase33_SHAREABLE.json
 - ICHI2027_Phase33_HELDOUT_CASE_RESULTS.csv
 - ICHI2027_Phase33_HELDOUT_EVENT_RESULTS.csv
